# 🖋️ chainmcp — the signing adapter

In `06_blockchain_from_zero.ipynb` you drove a chain by hand: raw web3.py, ABIs,
`call()` versus `transact()`, and `Settlement.sol` read section by section. It worked —
and it was fiddly. Nobody wants an agent (or a controller) juggling codecs and selector
tables. This chapter is where the repo's most failure-prone seam gets a keeper:
**chainmcp**, the adapter that plugs the raw chain into the tidy `EntitlementReader`
port from `03_protocols_and_ports.ipynb` — and the **only** package in the repo that
ever holds a private key.

**What you'll be able to do after this notebook**

- Build one `ChainClient` per agent and *prove* — with an AST scan of the whole repo — that key-loading power exists only inside `chainmcp` (rule 2)
- Explain what a digital signature buys you (authenticity, integrity, non-repudiation) and verify EIP-191 and EIP-712 signatures yourself
- Recompute the exact 32-byte digest the deployed contract computes — byte for byte, Python versus Solidity
- Buy ticket #7 for real: faucet → allowance → `approve_and_fulfill`, then assert all six on-chain effects and read the ticket back as a typed model
- Watch a revocation arrive on a background thread, decode the ticket's on-chain fine print, and debug the classic *BadSignature-with-everything-looking-right*

**You need:** notebooks 00–06. The signing and digest cells run with **no chain at
all**. The live cells want `anvil` on PATH (Foundry) and built artifacts
(`forge build --root contracts`, once) — without them they print a polite skip line,
exactly like 06.

**Estimated time:** 75–100 minutes.

> The compact live-lab tour of the same code is
> [`e2e/notebooks/chain_client_explore.ipynb`](../chain_client_explore.ipynb); this
> chapter is the slow, from-zero version.

## 1. Two chapters meet: the port from 03, the chain from 06

Recall the hexagon from 03: a **port** is a hole (a `Protocol` naming the methods
someone needs), an **adapter** is a plug (a class that really implements them). In 05
the `EntitlementReader` hole was filled by cardboard — `FakeChain`, a dict. Chapter 06
gave you the real thing behind the hole, but raw. chainmcp is the plug: it wraps every
web3 incantation from 06 behind the port's five clean methods, plus this agent's writes.

First, meet the package from the outside. Its `__init__.py` is a **facade**: it
re-exports the public names into one flat namespace (`from chainmcp import ChainClient`,
never `from chainmcp.client import ...`) and deliberately leaves the lab plumbing
(`chainmcp.testing`) out — production imports can't *accidentally* pull in code that
spawns chain processes.

In [ ]:
from pathlib import Path

# repo root: walk up to the folder holding .git (this notebook's cwd is 3 levels deep)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

import chainmcp

print("the public surface, one flat namespace →")
for name in chainmcp.__all__:
    print("  ", name)

import chainmcp.reader

assert chainmcp.ChainReader is chainmcp.reader.ChainReader  # re-export = same object, not a copy
assert "launch_anvil" not in dir(chainmcp)                  # lab plumbing is opt-in, never ambient
print("\n✓ two classes, one exception, seven signing names — and NO test plumbing")

## 2. Rule 2, machine-checked: who can sign, who can only verify

From 06: whoever knows a private key **is** that identity — a leaked key isn't a
password to reset, it's the agent's whole self, gone. So the repo's scariest bug class
is a key drifting into a log line, a return value, another package. Hard rule 2 exists
for exactly this: *private keys live only in chainmcp*.

But a rule in a README protects nothing. Let's check it against the code itself, using
`ast` — Python's own parser, which reads a `.py` file into a tree of nodes you can walk
(you've imported modules since 00; now we read them *as data*). The thing to hunt for is
`Account.from_key(...)`: the one call that turns a key string into signing power.
Watch the split in the output: `eth_account` shows up in the controller too — but only
for `recover`, which needs **no key**. That's rule 2's second sentence, visible:
*"the controller verifies signatures; it never signs."*

In [ ]:
import ast

def loads_a_key(tree):  # any `Account.from_key(...)` — string → signing power
    return any(
        isinstance(n, ast.Attribute) and n.attr == "from_key"
        and isinstance(n.value, ast.Name) and n.value.id == "Account"
        for n in ast.walk(tree)
    )

def imports_eth_account(tree):
    for node in ast.walk(tree):
        mods = ([a.name for a in node.names] if isinstance(node, ast.Import)
                else [node.module] if isinstance(node, ast.ImportFrom) and node.module else [])
        if any(m == "eth_account" or m.startswith("eth_account.") for m in mods):
            return True
    return False

signers, verifiers = [], []
for py in sorted(ROOT.glob("*/src/**/*.py")):           # every package's source, whole repo
    tree = ast.parse(py.read_text())
    if loads_a_key(tree):
        signers.append(str(py.relative_to(ROOT)))
    elif imports_eth_account(tree):
        verifiers.append(str(py.relative_to(ROOT)))

print("can LOAD a key (Account.from_key):")
for f in signers:
    print("   ", f)
print("touch eth_account but never load a key (verify-only):")
for f in verifiers:
    print("   ", f)

assert all(f.startswith("chainmcp/") for f in signers)  # rule 2, checked by machine
assert "controller/src/controller/auth.py" in verifiers # the bouncer verifies, never signs (08)
print("\n✓ signing power exists in chainmcp and nowhere else")

**✏️ Your turn 2.1 — scan for the chain itself**

Rule 2 has a quieter sibling: chainmcp should also be the only package that *talks to
the chain* at all. Adapt the scan to hunt for imports of `web3` instead of
`eth_account`. **Predict first** (write it as a comment): which packages will show up?
Success: your commented-out assert passes when un-commented.

In [ ]:
# my prediction: the packages that import web3 are ...   ← TODO write your guess BEFORE running

web3_importers = []
for py in sorted(ROOT.glob("*/src/**/*.py")):
    tree = ast.parse(py.read_text())
    for node in ast.walk(tree):
        mods = ([a.name for a in node.names] if isinstance(node, ast.Import)
                else [node.module] if isinstance(node, ast.ImportFrom) and node.module else [])
        if any(m == "web3" or m.startswith("web3.") for m in mods):
            web3_importers.append(str(py.relative_to(ROOT)))
            break

for f in web3_importers:
    print("  ", f)
# assert all(f.startswith("chainmcp/") for f in web3_importers) and len(web3_importers) == 3

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
web3_importers = []
for py in sorted(ROOT.glob("*/src/**/*.py")):
    tree = ast.parse(py.read_text())
    for node in ast.walk(tree):
        mods = ([a.name for a in node.names] if isinstance(node, ast.Import)
                else [node.module] if isinstance(node, ast.ImportFrom) and node.module else [])
        if any(m == "web3" or m.startswith("web3.") for m in mods):
            web3_importers.append(str(py.relative_to(ROOT)))
            break
assert all(f.startswith("chainmcp/") for f in web3_importers) and len(web3_importers) == 3
```

Exactly three files, all inside chainmcp (`client.py`, `reader.py`, `testing.py`) — the
rest of the repo reaches the chain only *through* this adapter, never around it.
</details>

## 3. A throwaway chain, pinned to story time (and this notebook's guard)

Same guard discipline as 06: two booleans decide whether the live cells run. If `anvil`
or the forge artifacts are missing, every chain cell prints a skip line instead of
crashing — and the pure-signing cells still run, because cryptography needs no chain.

`launch_anvil` starts a disposable chain and deploys **MockTOK first, then
A2ASettlement** — same order as the deploy script, so MockTOK lands at the exact address
`fixtures.MOCK_TOK` hardcodes (04's one-source-of-truth discipline, kept by deploy
order). We pin the clock to ~13:32 *story time*: the canonical offer's quote expires at
14:20 story time, and **chain time is the only clock that counts** (ADR-004) — your
wall clock is long past 15 Sep 2025; the chain's isn't, so the offer is alive here.

One gotcha worth reading twice: `launch_anvil` returns its addresses in
`anvil.deployment` — always pass `deployment=anvil.deployment` when you build clients,
or they'd silently read a stale checked-in address file and every call would revert
confusingly.

In [ ]:
from a2a_interfaces import fixtures as fx
from chainmcp.testing import ANVIL_KEYS, anvil_available, artifacts_available, launch_anvil

CHAIN = anvil_available() and artifacts_available()
SKIP = ("skipped: needs anvil + forge artifacts — install Foundry "
        "(curl -L https://foundry.paradigm.xyz | bash && foundryup), then: forge build --root contracts")

STORY_TIME = fx.WINDOW.start - 1680      # 13:32 story time — 28 min before Ada's window opens

anvil = None
if CHAIN:
    anvil = launch_anvil(timestamp=STORY_TIME)
    print("rpc        →", anvil.rpc_url, " ← a random free port; never hardcode it")
    print("deployment →", anvil.deployment)
    assert anvil.deployment["MockTOK"] == fx.MOCK_TOK   # deploy order kept 04's promise
    print("\n✓ a disposable chain, frozen at story time, MockTOK exactly where 04 said")
else:
    print(SKIP)

## 4. One client per identity — and a keyless twin for the controller

The custody rule has a *shape*: **one `ChainClient` per identity**, constructed *with*
that identity's key. From that moment the key lives inside the object — callers see the
derived `.address` and finished signatures, never the key itself. The keys below are
Anvil's four well-known dev accounts (public constants, not secrets — but rule 2 applies
to the pattern, so even these literals live in `chainmcp.testing` and nowhere else):
account #0 *is* Ada, #1 *is* Bell, #2 Carol — and #3 Mallory, deliberately penniless.

The first thing to check about any adapter: does it fit the hole? Same
structural-typing beat as every mock/real pair in this course.

One micro-construct before the code: `lambda arg: expression` is an *inline, unnamed
function* — `make = lambda key: ChainClient(key, …)` means exactly
`def make(key): return ChainClient(key, …)`, squeezed onto one line for a helper too
small to deserve a `def`. You'll meet it again as `fmt` in §10.

In [ ]:
from a2a_interfaces import EntitlementReader
from chainmcp import ChainClient

if not CHAIN:
    print(SKIP)
else:
    make = lambda key: ChainClient(anvil.rpc_url, key, deployment=anvil.deployment, poll_interval=0.2)
    ada, bell, carol, mallory = (make(ANVIL_KEYS[n]) for n in ("ada", "bell", "carol", "mallory"))

    print("Ada     →", ada.address)
    print("Bell    →", bell.address)
    print("Carol   →", carol.address)
    print("Mallory →", mallory.address)
    assert ada.address == fx.ADA and bell.address == fx.BELL   # key → address, exactly as 06 taught

    assert isinstance(ada, EntitlementReader)
    print("\nisinstance(ada, EntitlementReader) → True  ← the REAL adapter fits 05's cardboard hole")

Now the twin. In chapter 08 the **controller** will need to *read* the chain — owners,
windows, revocations — but rule 2's spirit says it must never hold signing power. Enter
`ChainReader`: `ChainClient` with the write half **amputated**. Not hidden, not
subclassed away — *absent*.

It reuses ChainClient's read methods through a trick you haven't met: **method borrowing
by class-body assignment**. In 01 you learned functions are objects (that's what made
decorators possible). So a class may simply *assign* another class's function into its
own body — same function object, now bound to whatever `self` calls it. Toy first:

In [ ]:
class Greeter:
    def hello(self):
        return "hi, I am " + self.name

class Borrower:
    hello = Greeter.hello        # a function is an object; assigned here, it becomes OUR method

b = Borrower()
b.name = "the borrower"
print(b.hello(), " ← Greeter's code ran with Borrower's self")
assert Borrower.hello is Greeter.hello                  # the SAME object, not a copy
print("✓ borrowing shares code without inheriting the rest of the class")

Why borrow instead of inherit? Inheritance would drag the **whole** ChainClient in —
`sign_offer`, `faucet`, the `_acct` attribute — merely *unused*, still reachable. With
borrowing, ChainReader's `__init__` never creates an account object at all: there is no
code path, not even a dormant one, from a controller to a key. Read the real class —
it's short — then check three claims against it: the borrowed methods are literally the
same objects, the constructor has no key parameter, and the write half is *gone*, not
hidden.

In [ ]:
import inspect
from chainmcp import ChainReader

print(inspect.getsource(ChainReader))

assert ChainReader.owner_of is ChainClient.owner_of        # borrowed, not copied
print("borrowed method is ChainClient's own  →", ChainReader.get is ChainClient.get)

print("constructor →", inspect.signature(ChainReader.__init__), " ← no private_key parameter EXISTS")
assert "private_key" not in inspect.signature(ChainReader.__init__).parameters

leaks = [n for n in dir(ChainReader) if "sign" in n or "fulfill" in n or "faucet" in n]
print("write-side names on ChainReader →", leaks, " ← amputated, not underscored")
assert leaks == []

And the live proof: a reader built with **no key argument at all** satisfies the same
port, and reads the same chain. `chain_time()` should be exactly the timestamp we
pinned — nothing has mined a block yet, and the chain's clock moves only when blocks do.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    reader = ChainReader(anvil.rpc_url, deployment=anvil.deployment, poll_interval=0.2)

    assert isinstance(reader, EntitlementReader)
    print("isinstance(reader, EntitlementReader) → True   ← same port, zero keys")

    t = reader.chain_time()
    print("reader.chain_time() →", t, " ← == STORY_TIME:", t == STORY_TIME)
    assert t == STORY_TIME     # frozen until a transaction mines a block (ADR-004)

**✏️ Your turn 4.1 — try to smuggle the write half back in**

The borrowing trick cuts both ways: you can call `ChainClient.sign_offer` with the
*reader* as `self`. Predict the exception **type** this raises (write it as a comment),
then run. Success: the error name tells you exactly which missing ingredient stops a
keyless object from signing.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: the exception type will be ...        ← TODO write it BEFORE running
    try:
        ChainClient.sign_offer(reader, fx.CANONICAL_OFFER)   # borrowed write method, reader as self
        print("✗ it signed?! custody is broken")
    except Exception as err:
        print("→", type(err).__name__, ":", err)
        # assert type(err).__name__ == "AttributeError"     # un-comment once you've predicted

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
try:
    ChainClient.sign_offer(reader, fx.CANONICAL_OFFER)
except AttributeError as err:
    print(err)          # 'ChainReader' object has no attribute '_acct'
```

`sign_offer` needs `self._acct` — the account object built from the key — and a
`ChainReader` simply never made one, so even ChainClient's own code cannot sign through
it: custody by *absence*, not by discipline.
</details>

## 5. What a signature gives you — EIP-191 first

You met **hashes** in 04: a fingerprint that proves *what* the data was. A **digital
signature** proves *who said it*: a small blob (65 bytes here) computed from a message
**and** a private key. Anyone can take message + signature and *recover* the signer's
address — no key needed to check. That buys three things, each load-bearing for this
project:

- **authenticity** — only the key holder could have produced it (Bell really made this offer);
- **integrity** — it covers every byte; change one and verification points at a stranger;
- **non-repudiation** — Bell can't later deny signing (anyone can re-verify, forever).

Two signature flavors appear in this repo, and we start with the simpler: **EIP-191**
("personal_sign") — sign a plain string, tagged with a prefix so it can never be
mistaken for a transaction. `encode_defunct` builds that tagged message. Note what's
*absent* below: no chain, no anvil, no guard — cryptography is pure math.

In [ ]:
from eth_account import Account
from eth_account.messages import encode_defunct

ada_acct = Account.from_key(ANVIL_KEYS["ada"])      # key → account object (key → address was 06)
assert ada_acct.address == fx.ADA

note = "Bell, activate my ticket please. — Ada"
sig = ada_acct.sign_message(encode_defunct(text=note)).signature
print("signature →", sig.hex()[:32], "…  (" + str(len(sig)) + " bytes — always 65)")

whodunit = Account.recover_message(encode_defunct(text=note), signature=sig)
print("recovered →", whodunit, " ✓ Ada" if whodunit == fx.ADA else " ✗")
assert whodunit == fx.ADA and len(sig) == 65

impostor = Account.recover_message(encode_defunct(text=note + "!"), signature=sig)
print("tampered  →", impostor, " ← ONE added character: a stranger, not an error")
assert impostor != fx.ADA

That last line is the sharpest fact in this chapter: **verification never "fails
loudly" — it just recovers a different address.** Keep it in mind for §16.

The repo's real EIP-191 use is the **activation proof** (docs/03 §3.2): buying ticket #7
proved *payment*; activating the service must prove *ownership, live, to the
controller*. The controller sends a challenge containing a **nonce** — a
use-once value that makes each challenge fresh (chapter 08 owns the full
challenge–response dance) — and the consumer signs the filled-in template below.
`ChainClient.sign_activation_proof` is that one move with the agent's own key; the
controller will verify it in pure Python, no contract involved.

In [ ]:
from chainmcp.signing import ACTIVATION_PROOF_TEMPLATE, activation_proof_message

print("template →", ACTIVATION_PROOF_TEMPLATE, "\n")
print(inspect.getsource(ChainClient.sign_activation_proof))

message = activation_proof_message("bw-ctrl-1", "0xabcd", fx.TICKET_ID, fx.WINDOW.start + 300)
print("filled   →", message, " ← a readable string, not opaque hex")

proof_sig = ada_acct.sign_message(encode_defunct(text=message)).signature
recovered = Account.recover_message(encode_defunct(text=message), signature=proof_sig)
print("recovers →", recovered, " ✓ Ada")
assert recovered == fx.ADA

**✏️ Your turn 5.1 — replay the proof with a different nonce**

An attacker records Ada's `proof_sig` off the wire and re-sends it against a **new**
challenge (nonce `0xd00d` instead of `0xabcd`) — a **replay attack**, the classic reason
nonces exist. Predict what `recover_message` returns (Ada? an error? a stranger?), write
the prediction as a comment, then run. Success: the un-commented assert passes.

In [ ]:
# my prediction: recovering against the new-nonce message yields ...   ← TODO write it first

replayed = activation_proof_message("bw-ctrl-1", "0xd00d", fx.TICKET_ID, fx.WINDOW.start + 300)
who = Account.recover_message(encode_defunct(text=replayed), signature=proof_sig)
print("recovered →", who)
print("Ada is    →", fx.ADA)
# assert who != fx.ADA           # un-comment once you've predicted

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
replayed = activation_proof_message("bw-ctrl-1", "0xd00d", fx.TICKET_ID, fx.WINDOW.start + 300)
who = Account.recover_message(encode_defunct(text=replayed), signature=proof_sig)
assert who != fx.ADA
```

The signature covers every byte of the *original* string; against different bytes it
recovers a stranger, the controller's `ownerOf` comparison fails, and the replay dies —
integrity plus a fresh nonce is the whole defense.
</details>

## 6. EIP-712 from zero: the domain that binds a signature to one world

Offers are structs, not strings — twelve fields. You *could* personal_sign their JSON,
but then (a) a human's wallet shows only opaque hex to approve blind, and (b) a Solidity
contract would have to re-serialize JSON byte-identically to verify — hopeless. So the
ecosystem grew **EIP-712, typed-data signing**: sign a *struct* under agreed field names
and types. Wallets can render exactly what's being signed ("provider: 0x7099…, price:
10000000000000000000"), and a contract can rebuild the identical digest from calldata.

Every EIP-712 signature is bound to a **domain** — four fields that pin *whose*
signature this can ever be:

| field | value here | what it kills |
|---|---|---|
| `name` | `"A2AProvisioning"` | signatures from other protocols |
| `version` | `"0"` | signatures from other *versions* of this protocol |
| `chainId` | `31337` | **cross-chain replay** — a mainnet signature is dead on our anvil, and vice versa |
| `verifyingContract` | the settlement's address | **cross-contract replay** — a signature for contract A is dead at contract B |

`name`/`version` are pinned by docs/03 §2.1 and must match the contract **byte for
byte** — watch both sides say the same two strings:

In [ ]:
from chainmcp.signing import eip712_domain

print(inspect.getsource(eip712_domain))

PRETEND_SETTLEMENT = "0x" + "5e" * 20     # any address works for practice; the REAL one arrives in §9
domain = eip712_domain(31337, PRETEND_SETTLEMENT)
print("python side   →", domain)

SOL = (ROOT / "contracts/src/Settlement.sol").read_text()
ctor = next(ln.strip() for ln in SOL.splitlines() if "constructor" in ln and "EIP712(" in ln)
print("solidity side →", ctor)

assert domain["name"] == "A2AProvisioning" and domain["version"] == "0"   # docs/03 §2.1
assert 'EIP712("A2AProvisioning", "0")' in ctor
print("\n✓ same two strings in both languages — nobody retypes these casually")

**✏️ Your turn 6.1 — kill a cross-chain replay yourself**

Compute the canonical offer's digest twice: once for our lab chain (`chainId=31337`) and
once for Ethereum mainnet (`chainId=1`), same pretend contract. Predict first: same 32
bytes, or different? Success: the un-commented assert settles it.

In [ ]:
from chainmcp import offer_digest

# my prediction: the two digests will be ...   ← TODO same or different? write it first
lab_digest     = offer_digest(fx.CANONICAL_OFFER, 31337, PRETEND_SETTLEMENT)
mainnet_digest = offer_digest(fx.CANONICAL_OFFER, 1, PRETEND_SETTLEMENT)

print("chainId 31337 → 0x" + lab_digest.hex())
print("chainId 1     → 0x" + mainnet_digest.hex())
# assert lab_digest != mainnet_digest          # un-comment once you've predicted

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
assert offer_digest(fx.CANONICAL_OFFER, 31337, PRETEND_SETTLEMENT) != \
       offer_digest(fx.CANONICAL_OFFER, 1, PRETEND_SETTLEMENT)
```

Different — the chainId sits *inside* the domain separator, so the very bytes being
signed differ per chain: a signature captured on one chain recovers to a stranger on
another. That's the cross-chain replay defense, for free.
</details>

## 7. The typehash: field order is part of the signature

The struct half of EIP-712 starts from a **type string** — the struct's name, fields and
types, in declaration order, as one exact string. Its keccak is the **typehash**, and it
is mixed into every digest, so *field order is part of what gets signed*. eth-account
derives the type string from `OFFER_TYPES`, the module constant in
`chainmcp/signing.py` — a dict whose ordering is as load-bearing as source code.

The claim to pin: the string Python derives appears **verbatim** inside
`Settlement.sol`'s `OFFER_TYPEHASH`. One reordered word in either place and every
signature recovers to a stranger.

In [ ]:
from chainmcp.signing import OFFER_TYPES
from eth_utils import keccak

print("field order →", [f["name"] for f in OFFER_TYPES["Offer"]], "\n")

type_string = "Offer(" + ",".join(f["type"] + " " + f["name"] for f in OFFER_TYPES["Offer"]) + ")"
print("type string →", type_string, "\n")

TYPEHASH = keccak(text=type_string)
print("typehash    → 0x" + TYPEHASH.hex())

assert type_string in SOL       # the EXACT same literal sits inside Settlement.sol
print("\n✓ this string appears verbatim in Settlement.sol — one spelling, two compilers")

One more rule hides inside the struct hash, and it is **the** EIP-712 rule that bites
everyone: **dynamic-length fields don't enter raw** — a `bytes` field (our `params`
blob) is replaced by its keccak256 *inside* the struct hash. Fixed-size fields
(`address`, `uint64`, `bytes32`) go in as-is; anything variable-length goes in as its
hash. Forget this on one side of the seam and the digests silently diverge.

Read the contract's `hashOffer` — the Solidity twin of what eth-account does — and find
the one field that gets wrapped:

In [ ]:
start = SOL.index("function hashOffer")
end = SOL.index("\n    }", start) + len("\n    }")
print(SOL[start:end])

assert "keccak256(offer.params)" in SOL[start:end]
print("\n✓ `params` (dynamic bytes) enters as its keccak — every other field goes in raw")

**✏️ Your turn 7.1 — reorder two fields, watch the typehash change**

Swap `price` and `validUntil` in a **copy** of the field list and rebuild the type
string. Predict first: does the typehash change, even though the *set* of fields is
identical? Success: the un-commented assert passes.

In [ ]:
# my prediction: same typehash / different typehash      ← TODO write it first
fields = list(OFFER_TYPES["Offer"])
fields[8], fields[9] = fields[9], fields[8]          # price ↔ validUntil: same fields, new ORDER

reordered = "Offer(" + ",".join(f["type"] + " " + f["name"] for f in fields) + ")"
print("real      → 0x" + TYPEHASH.hex()[:32] + "…")
print("reordered → 0x" + keccak(text=reordered).hex()[:32] + "…")
# assert keccak(text=reordered) != TYPEHASH          # un-comment once you've predicted

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
assert keccak(text=reordered) != TYPEHASH
```

Different — the type string is hashed as *text*, so order changes the bytes. This is why
`OFFER_TYPES`' comment calls its ordering "load-bearing": a well-meaning alphabetical
re-sort of that dict would break every signature in the system, with no error message
anywhere.
</details>

## 8. From wire shape to signable bytes

The `Offer` model from 02/04 is a **wire shape**: hex *strings* for the byte fields, a
decimal *string* for the price (JSON can't hold 256-bit ints — 04). eth-account wants
machine types: real `bytes`, real `int`s. `offer_to_message` is that converter — and
note `model_dump(by_alias=True)`: the camelCase aliases from 02 are exactly the
contract's field names, because *what is signed must equal what the contract verifies*.

In [ ]:
from chainmcp.signing import offer_to_message

print(inspect.getsource(offer_to_message))

msg = offer_to_message(fx.CANONICAL_OFFER)
print({k: type(v).__name__ for k, v in msg.items()})

assert isinstance(msg["price"], int) and isinstance(msg["salt"], bytes)
assert "serviceType" in msg                      # camelCase alias, not snake_case
print("\n✓ camelCase keys, ints for uints, raw bytes for bytes32 — what eth-account wants")

**✏️ Your turn 8.1 — which fields does the converter leave alone?**

`offer_to_message` converts *only* what eth-account can't take as-is. Predict the
Python type of `msg["provider"]` and of `msg["startTime"]` (write both as a comment),
then run. Success: the un-commented assert passes — and you can say why neither field
needed converting.


In [ ]:
# my prediction: msg["provider"] is a ...  msg["startTime"] is a ...   ← TODO write both first
print("provider  →", type(msg["provider"]).__name__.ljust(4), "=", msg["provider"])
print("startTime →", type(msg["startTime"]).__name__.ljust(4), "=", msg["startTime"])
# assert isinstance(msg["provider"], str) and isinstance(msg["startTime"], int)   # un-comment once predicted


<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
assert isinstance(msg["provider"], str) and isinstance(msg["startTime"], int)
```

The converter touches only the four hex-string byte fields (→ raw `bytes`) and the
decimal-string price (→ `int`). An address stays a plain string — eth-account reads
`0x…` addresses as-is — and the `uint64` times were already real ints on the wire,
because they fit in a JSON number; only 256-bit `price` didn't (04).
</details>


And the final assembly, `offer_digest`: EIP-712's closing move is
`keccak(0x19 ‖ 0x01 ‖ domainSeparator ‖ structHash)` — the two magic bytes prefix says
"this is typed data, never a transaction". The result is **32 bytes, and those 32 bytes
are ALL a signature ever commits to** — which is why the two stacks agreeing on them is
the whole game.

In [ ]:
from chainmcp import offer_digest

print(inspect.getsource(offer_digest))

local = offer_digest(fx.CANONICAL_OFFER, 31337, PRETEND_SETTLEMENT)
print("digest → 0x" + local.hex(), " ← " + str(len(local)) + " bytes")
assert len(local) == 32

## 9. The cross-stack assert — milestone M1.5 in one cell

Everything above was the Python half of the seam. The Solidity half — `hashOffer` — has
been computing "the same" digest since 06. Time for the confrontation this whole
milestone existed for: **ask the live contract for its digest of the canonical offer and
compare byte-for-byte with Python's**. If any of the pieces you just built — domain,
type string, field order, dynamic-hashing, wire conversion — diverged by one byte, this
cell fails. (It's also a pinned CI test: `chainmcp/tests/test_cross_stack.py`.)

Names with a leading `_` are internal — we peek at `ada._settlement` only to ask the
contract directly, the way that test does.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    python_digest = ada.offer_digest(fx.CANONICAL_OFFER)
    contract_digest = ada._settlement.functions.hashOffer(ada._offer_tuple(fx.CANONICAL_OFFER)).call()

    print("python   → 0x" + python_digest.hex())
    print("solidity → 0x" + contract_digest.hex())
    assert python_digest == contract_digest

    assert ada._settlement.functions.OFFER_TYPEHASH().call() == TYPEHASH   # §7, confirmed on-chain
    print("\n✓ two languages, one hash — the seam is sound, byte for byte")

Now the signature itself. `sign_offer` is **off-chain**: no transaction, no gas — the
chain has no idea Bell promised anything. The 65-byte signature *is* the promise, and
anyone holding it can check who made it:

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    from chainmcp import encode_offer

    signed = bell.sign_offer(fx.CANONICAL_OFFER, terms_doc=fx.TERMS_DOC)
    raw = bytes.fromhex(signed.signature[2:])
    print("signature →", signed.signature[:26] + "…  (" + str(len(raw)) + " bytes — the whole promise)")

    recovered = Account.recover_message(
        encode_offer(fx.CANONICAL_OFFER, anvil.deployment["chainId"], anvil.deployment["A2ASettlement"]),
        signature=signed.signature,
    )
    print("recovers  →", recovered, " ✓ Bell" if recovered == fx.BELL else " ✗")
    assert len(raw) == 65 and recovered == fx.BELL

**✏️ Your turn 9.1 — the wrong signer**

Carol signs the canonical offer — an offer whose `provider` field says **BELL**. Predict
which *address* the recovery yields: Bell? Carol? A stranger? Write your prediction,
run, then un-comment the assert.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: it recovers to ...          ← TODO write it BEFORE running
    forged = carol.sign_offer(fx.CANONICAL_OFFER)      # Carol signs; the offer names Bell as provider
    recovered = Account.recover_message(
        encode_offer(fx.CANONICAL_OFFER, anvil.deployment["chainId"], anvil.deployment["A2ASettlement"]),
        signature=forged.signature,
    )
    print("recovers →", recovered)
    print("Carol is →", carol.address, " · Bell is →", fx.BELL)
    # assert recovered == carol.address and recovered != fx.BELL   # un-comment once predicted

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
assert recovered == carol.address and recovered != fx.BELL
```

Recovery always names the *actual* signer — Carol. `fulfill` compares the recovered
address against `offer.provider` and would refuse with `BadSignature` (§12): you cannot
issue promises in Bell's name without Bell's key. That's authenticity doing its job.
</details>

## 10. Money on a leash: faucet, and allowance from zero

Before anyone buys, the cast needs TOK — 06's ERC-20 lab currency, minted freely by the
faucet because the *mechanics* are the lesson here, not scarcity. Ada gets buying money;
Carol gets enough for six tickets (you'll see why in §11); **Mallory deliberately gets
nothing** — she's §12's broke buyer.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    ada.faucet(50 * 10**18)          # Ada: buying money
    carol.faucet(60 * 10**18)        # Carol: exactly six tickets' worth
                                     # Mallory: nothing, on purpose
    fmt = lambda wei: str(wei / 10**18) + " TOK"
    for name, who in [("Ada", ada), ("Bell", bell), ("Carol", carol), ("Mallory", mallory)]:
        print(name.ljust(8), "→", fmt(ada.tok_balance(who.address)))

    assert ada.tok_balance(ada.address) == 50 * 10**18
    assert ada.tok_balance(mallory.address) == 0

Now the idea the whole purchase stands on: **allowance**. An ERC-20 contract cannot
reach into Ada's balance — only Ada's own transactions move Ada's money. So how can the
*settlement contract* take her 10 TOK during `fulfill`? Answer: payment is
**pull-based**. Ada first calls `approve(settlement, amount)` on the *token* contract —
a standing permission, "the settlement may pull up to this much from me" — and later the
settlement calls `transferFrom` to pull it. Think of it as signing a one-time direct-debit
authorization, not handing over cash.

Watch the dance by hand (peeking at `ada._tok`, the token contract object, and
`ada._transact`, the internal send-and-wait helper — internals, flagged as always):

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    settlement = ada._settlement.address
    def ada_allowance():             # how much the settlement may currently pull from Ada
        return ada._tok.functions.allowance(ada.address, settlement).call()

    print("allowance now         →", ada_allowance())
    ada._transact(ada._tok.functions.approve(settlement, 3 * 10**18))
    print("after approve(3 TOK)  →", fmt(ada_allowance()), " ← a permission, NOT a payment")
    print("Ada's balance         →", fmt(ada.tok_balance(ada.address)), " ← unchanged: no money moved")

    ada._transact(ada._tok.functions.approve(settlement, 0))
    print("after approve(0)      →", ada_allowance(), " ← permission withdrawn")
    assert ada_allowance() == 0 and ada.tok_balance(ada.address) == 50 * 10**18

**✏️ Your turn 10.1 — does `approve` add up, or overwrite?**

Ada approves 3 TOK, then — without spending any of it — approves 5 TOK. Predict the
allowance after both calls: 8 TOK or 5 TOK? Write it first, run, un-comment the
assert. Success: you know whether `approve` *sets* or *adds* — a classic ERC-20
gotcha.


In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: after approve(3) then approve(5), the allowance is ... TOK   ← TODO write it first
    ada._transact(ada._tok.functions.approve(settlement, 3 * 10**18))
    ada._transact(ada._tok.functions.approve(settlement, 5 * 10**18))
    print("allowance after both →", fmt(ada_allowance()))
    # assert ada_allowance() == 5 * 10**18        # un-comment once you've predicted
    ada._transact(ada._tok.functions.approve(settlement, 0))   # tidy up — §11 starts from a clean slate


<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
assert ada_allowance() == 5 * 10**18   # approve SETS the allowance — it never adds
```

Each `approve` *overwrites* the standing permission — 5 replaces 3, no arithmetic.
That's also why `approve_and_fulfill` can approve exactly the price every time
without worrying about leftovers from earlier attempts.
</details>


## 11. The purchase: six effects, one transaction

A bit of stage-setting first, to make the story *literally* true: Ada's ticket is **#7**
because Bell sold six before hers (04's fixtures promised exactly that). Carol buys
those six — same terms, six **fresh salts**. The salt is the ticket-stub serial from 04:
same promise, different serial ⇒ different digest ⇒ needs its own signature from Bell.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    for i in range(1, 7):
        pre_sale = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + format(i, "064x")})
        _, eid = carol.approve_and_fulfill(bell.sign_offer(pre_sale))
        print("ticket #" + str(eid), "→ Carol   (salt …" + format(i, "04x") + ", fresh signature)")
        assert eid == i

**✏️ Your turn 11.1 — predict Carol's purse**

Carol fauceted 60 TOK and bought six tickets at the canonical price. Write your
prediction of her remaining balance as a comment, then run and un-comment the assert.
Success: prediction and chain agree.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: Carol now holds ... TOK        ← TODO write it BEFORE running
    print("Carol →", fmt(ada.tok_balance(carol.address)))
    # assert ada.tok_balance(carol.address) == 0    # un-comment once you've predicted

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
assert ada.tok_balance(carol.address) == 0
```

60 TOK − 6 tickets × 10 TOK = 0 — the canonical price really moved on every sale, and
balances are just contract storage you can always read back.
</details>

Now Ada's turn — the real thing, with the very `signed` offer Bell produced in §9.
`approve_and_fulfill` wraps the two-step dance: approve exactly the price, then call
`fulfill`, which you read in 06 — verify signature, pull payment, mint — **one
transaction, so either everything happens or nothing does** (invariant I3).

Six distinct effects should now be true. We assert every one:

1. the returned id is `fixtures.TICKET_ID` — **#7**;
2. Ada paid exactly 10 TOK;
3. Bell earned exactly 10 TOK;
4. `owner_of(7)` is Ada — the ERC-721 ticket exists and is hers;
5. the offer's digest is marked **consumed** — the single-use ledger (I2);
6. the allowance is back to 0 — pulled in full, nothing left dangling.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    before = {"ada": ada.tok_balance(ada.address), "bell": ada.tok_balance(bell.address)}

    tx_hash, ticket_id = ada.approve_and_fulfill(signed)

    paid = before["ada"] - ada.tok_balance(ada.address)
    earned = ada.tok_balance(bell.address) - before["bell"]
    print("tx", tx_hash[:20] + "…")
    print("1 minted id       →", ticket_id)
    print("2 Ada paid        →", fmt(paid))
    print("3 Bell earned     →", fmt(earned))
    print("4 owner_of(7)     →", ada.owner_of(7), "(Ada)")
    print("5 digest consumed →", ada.offer_consumed(fx.CANONICAL_OFFER))
    print("6 allowance left  →", ada_allowance())

    assert ticket_id == fx.TICKET_ID == 7
    assert paid == earned == 10 * 10**18
    assert ada.owner_of(7) == fx.ADA
    assert ada.offer_consumed(fx.CANONICAL_OFFER) is True
    assert ada_allowance() == 0
    print("\n✓ six effects, one purchase — atomic because fulfill is ONE transaction (I3)")

Last stop of the read side: `get(7)`. In 04 you dissected the `params` ABI blob **by
hand** — two right-aligned 32-byte words. The adapter now does that decode for you and
returns an `EntitlementView`: a typed, validated pydantic model, with `params` as a real
`BandwidthParams`. No web3 types leak past this point — this exact object is what the
controller's predicate will judge in 08. And it should equal, field for field, the
fixture you've trusted since 04:

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    from a2a_interfaces import BandwidthParams

    view = ada.get(7)
    print(view.model_dump_json(indent=2))

    assert isinstance(view.params, BandwidthParams)
    assert view.params.capacity_bps == 50_000_000
    assert view == fx.CANONICAL_ENTITLEMENT_VIEW    # 04's hand-built fixture — now minted for real
    print("\n✓ the blob you decoded by hand in 04, decoded by the adapter — and it matches the fixture")

## 12. The chain says no: replay and tamper, decoded

Deny paths are where enforcement lives. In 05, `FakeChain` raised Python exceptions with
names like `OfferAlreadyUsed`; the real contract refuses with **custom errors** — and
`ChainClient` decodes them back into a `ChainRevert` exception carrying the same
`.name`, using a table built from the ABI: each error's 4-byte **selector** (a keccak
fingerprint — 04's trick again) mapped to its name.

Cheat #1: **replay**. Redeem the *same* signed offer twice. The contract keeps a ledger
keyed by the offer's digest (`consumed`), so the second attempt dies — the single-use
invariant I2. We also verify the raw selector by hand, so the decoding is no mystery:

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    from chainmcp import ChainRevert

    try:
        ada.approve_and_fulfill(signed)              # the SAME signed offer as §11
        print("✗ replay went through?!")
    except ChainRevert as err:
        selector = "0x" + keccak(text="OfferAlreadyUsed()")[:4].hex()
        print("chain said →", err.name)
        print("raw data   →", err.data[:10], " ← the revert's 4-byte selector")
        print("by hand    →", selector, " = keccak('OfferAlreadyUsed()')[:4]")
        assert err.name == "OfferAlreadyUsed" and err.data[:10] == selector

Cheat #2: **tamper**. Nudge ONE field of the signed offer by the smallest possible
amount — the price, up by 1 *wei* of TOK (0.000000000000000001 TOK). The digest of the
tampered offer no longer matches what Bell signed, recovery points at a stranger, and
the stranger isn't `offer.provider` → `BadSignature`.

This is the demo that matters for chapter 10: agents will ship offers to each other over
A2A messages — *any* wire, hostile or not. **Nothing in transit can be altered without
the signature breaking**, so the wire needs no trust at all. And note the aftermath: a
refused purchase leaves *no trace* — `approve_and_fulfill` withdraws its own allowance
when fulfill refuses.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    cheaper = signed.offer.model_copy(update={"price": str(int(signed.offer.price) + 1)})
    try:
        ada.approve_and_fulfill(signed.model_copy(update={"offer": cheaper}))
        print("✗ tampered offer went through?!")
    except ChainRevert as err:
        print("chain said →", err.name, " ← ONE field, nudged by one wei")
        assert err.name == "BadSignature"

    assert ada.tok_balance(ada.address) == before["ada"] - 10 * 10**18   # both cheats: no trace
    assert ada_allowance() == 0                                          # allowance auto-withdrawn
    print("✓ balances untouched, allowance withdrawn — refused purchases leave no trace")

**✏️ Your turn 12.1 — the broke buyer**

Mallory (0 TOK, remember) tries to buy a *fresh* offer from Bell (new salt, so no
replay in the way). Predict the error **name** first — is it one of ours
(`OfferExpired`, `BadSignature`, …) or one the token contract itself raises? Success:
the name in the output matches your un-commented assert.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: the error name will be ...        ← TODO write it BEFORE running
    broke_offer = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + format(99, "064x")})
    try:
        mallory.approve_and_fulfill(bell.sign_offer(broke_offer))
        print("✗ it went through?!")
    except ChainRevert as err:
        print("chain said →", err.name)
        # assert err.name == "ERC20InsufficientBalance"   # un-comment once you've predicted

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
assert err.name == "ERC20InsufficientBalance"
```

The refusal comes from **MockTOK**, not the settlement: approving costs nothing (it's
just permission), but when `fulfill` tries to *pull* 10 TOK from a 0-TOK purse, the
token contract reverts. The selector table was built from *both* ABIs precisely so
inherited OpenZeppelin errors decode by name too.
</details>

## 13. The watcher: revocation arrives on a background thread

Story chapter 8: Bell can pull the kill switch on a live ticket, and the controller must
tear the session down **mid-window** — not at the next poll of some human. The
mechanism is `watch_revoked(callback)`: a daemon thread (01's threading, now in
production posture) polls the chain every `poll_interval` seconds (0.2 here) for
`Revoked(id)` events and hands each id to your callback.

Two designed-in honesty points before you run it: delivery is **at-least-once** (a crash
mid-delivery re-delivers rather than skips — the right trade, because teardown is
idempotent, rule 8), and the baseline block is read *synchronously at registration*, so
an event mined a millisecond after your call can never fall in a gap.

In [ ]:
import time

if not CHAIN:
    print(SKIP)
else:
    seen = []
    ada.watch_revoked(seen.append)       # Ada's client watches — the OWNER learns her ticket died
    time.sleep(0.4)                      # let the watcher take its first look around

    bell.revoke(7)                       # the ISSUER pulls the kill switch, on-chain

    deadline = time.monotonic() + 10
    while not seen and time.monotonic() < deadline:
        time.sleep(0.05)

    print("watcher delivered →", seen)
    assert 7 in seen
    print("revoked flag →", ada.get(7).revoked, " · owner still", ada.owner_of(7)[:10] + "… ← no burn (I5)")
    assert ada.get(7).revoked is True and ada.owner_of(7) == fx.ADA

The flag flipped, the callback fired — and Ada still *owns* the ticket. Revocation is
one bit, never a burn: the token stays readable evidence of what was promised (I8).

And the clock, one more time (ADR-004): `chain_time()` is the **latest block's
timestamp** — the only clock any validity decision may consult. It never ticks on its
own: it changes only when a block mines. But watch the *size* of the jump below: anvil
stamps each new block with `pinned time + wall-seconds since launch` — it keeps an
offset from the moment §3 started it; it does **not** count blocks. So the drift is
roughly how long ago you ran §3, materialized in steps, one step per mined block.
Re-read `chain_time()` rather than caching it; every transaction publishes a fresh now.

> **Slow-session warning:** because the drift tracks your wall clock, a long
> interactive sitting eventually outruns the story: ~28 minutes after §3 the next
> block lands past `fx.WINDOW.start` (14:00), and ~48 minutes after §3 the canonical
> offer's `validUntil` passes — §11-style purchases would then revert with
> `OfferExpired`. Nothing is broken: restart the kernel and rerun from the top; §3
> pins a fresh chain back to 13:32.

In [ ]:
import datetime

if not CHAIN:
    print(SKIP)
else:
    t = ada.chain_time()
    stamp = datetime.datetime.fromtimestamp(t, datetime.UTC).strftime("%H:%M:%S, %d %b %Y")
    print("chain_time() →", t, " =", stamp, "story time")
    print("drift since launch →", t - STORY_TIME, "s  ← ≈ wall-seconds since §3, stamped when OUR blocks mined")
    assert t >= STORY_TIME               # never behind the pin; how far PAST it depends on session speed
    if t >= fx.WINDOW.start:
        print("⚠ slow session — story time has passed 14:00; restart the kernel and rerun to re-pin")

**✏️ Your turn 13.1 — can the owner pull the switch?**

Ada *owns* ticket #7. Predict what `ada.revoke(7)` does — success, or a named refusal?
Write the prediction, run, un-comment the assert. Success: you can explain *why* the
switch belongs to who it belongs to.

In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: ada.revoke(7) will ...          ← TODO write it BEFORE running
    try:
        ada.revoke(7)
        print("✗ the owner revoked her own ticket?!")
    except Exception as err:
        print("chain said →", getattr(err, "name", type(err).__name__))
        # assert err.name == "NotIssuer"             # un-comment once you've predicted

<details><summary>✅ Solution 13.1 — peek only after trying</summary>

```python
try:
    ada.revoke(7)
except ChainRevert as err:
    assert err.name == "NotIssuer"
```

The kill switch belongs to the party *bound by the promise* (the issuer, Bell), not its
beneficiary — Ada "revoking" her own ticket would let buyers un-promise sellers.
Bonus fact: Bell re-revoking #7 would *succeed* and re-fire the event — harmless,
because teardown is idempotent (rule 8) and delivery was at-least-once anyway.
</details>

## 14. The ticket's fine print: tokenURI, decoded

In 06 you met `tokenURI` as ERC-721's metadata hook. This contract's twist: the URI is
**self-contained** — a `data:application/json;base64,…` string rendered *from storage on
every call*. No web server, so it can never 404; rendered live, so it can never lie
(I7). We just revoked #7 — the fine print should already say so.

`base64` is plain-text armor for bytes (00 met the idea inside the ABI blob); Python's
`base64` module undoes it, and `json.loads` gives you the dict:

In [ ]:
import base64
import json

if not CHAIN:
    print(SKIP)
else:
    uri = ada.token_uri(7)
    print(uri[:64] + "…\n")

    meta = json.loads(base64.b64decode(uri.split(",", 1)[1]))
    print(json.dumps(meta, indent=2))

    assert meta["name"] == "A2A Entitlement #7"
    assert meta["revoked"] is True                   # the fine print flipped WITH the flag
    assert meta["issuer"] == fx.BELL.lower()         # Solidity hex-strings render lowercase
    print("\n✓ revoke the ticket and its own metadata says so — on-chain, un-404-able")

**✏️ Your turn 14.1 — read an innocent ticket's fine print**

Decode ticket **#1** (Carol's first pre-sale) the same way. Predict its `revoked` value
first. Success: name says `#1`, and the flag matches your prediction.

In [ ]:
import base64
import json

if not CHAIN:
    print(SKIP)
else:
    # my prediction: ticket #1's revoked flag is ...     ← TODO write it BEFORE running
    uri1 = ada.token_uri(1)
    meta1 = json.loads(base64.b64decode(uri1.split(",", 1)[1]))
    print(json.dumps(meta1, indent=2))
    # assert meta1["name"] == "A2A Entitlement #1" and meta1["revoked"] is False

<details><summary>✅ Solution 14.1 — peek only after trying</summary>

```python
uri1 = ada.token_uri(1)
meta1 = json.loads(base64.b64decode(uri1.split(",", 1)[1]))
assert meta1["name"] == "A2A Entitlement #1" and meta1["revoked"] is False
```

Only #7 was revoked; #1's metadata still reads `"revoked": false` — each ticket's fine
print is rendered from *its own* storage row, per call.
</details>

## 15. The same five moves, served as MCP tools

Everything you just did by hand — sign, fulfill, read, prove, faucet — is exactly what
an *agent* will need in chapter 10. Agents reach tools through **MCP** (Model Context
Protocol — a standard way to hand an LLM a set of callable tools; 10 owns the full
story). `chainmcp/mcp_server.py` is the custody rule made into a service: each agent
runs its **own** instance wrapping its **own** client, so the key stays in the process
and tool results carry only hashes, ids, signatures, addresses.

`chain_tools(client)` returns the five operations of docs/03 §6.1 as plain callables —
each one a **closure** that captured `client` (the same functions-holding-state trick
that powered 01's decorators). Dict in, dict out — and pydantic re-validates at the
boundary, so garbage never reaches the chain:

In [ ]:
import json

if not CHAIN:
    print(SKIP)
else:
    from chainmcp.mcp_server import chain_tools

    tools = chain_tools(bell)
    print("the five §6.1 operations →", sorted(tools))
    assert sorted(tools) == ["faucet", "fulfill_offer", "read_entitlement",
                             "sign_activation_proof", "sign_offer"]

    out = tools["read_entitlement"](7)
    print("read_entitlement(7) →", {k: out[k] for k in ("id", "issuer", "revoked")})

    assert ANVIL_KEYS["bell"][2:] not in json.dumps(out)   # custody: results NEVER carry the key

    try:
        tools["sign_offer"]({"provider": "not-an-address"})
    except Exception as err:
        print("garbage in →", type(err).__name__, " ← pydantic refuses at the boundary; the chain never saw it")

**✏️ Your turn 15.1 — the tools see what you did in §13**

These tools wrap *Bell's* live client, and §13 revoked ticket #7 (while #1, Carol's
first pre-sale, was left alone). Predict each ticket's `revoked` field as returned by
`tools["read_entitlement"]`, write both down, run, then un-comment the asserts.
Success: the plain-dict tool output agrees with everything you watched happen.


In [ ]:
if not CHAIN:
    print(SKIP)
else:
    # my prediction: #1 revoked = ...   #7 revoked = ...      ← TODO write both BEFORE running
    for tid in (1, 7):
        out = tools["read_entitlement"](tid)
        print("ticket #" + str(out["id"]), "→ revoked:", out["revoked"])
    # assert tools["read_entitlement"](1)["revoked"] is False   # un-comment once predicted
    # assert tools["read_entitlement"](7)["revoked"] is True


<details><summary>✅ Solution 15.1 — peek only after trying</summary>

```python
assert tools["read_entitlement"](1)["revoked"] is False
assert tools["read_entitlement"](7)["revoked"] is True
```

Same chain, same storage — the MCP wrapper adds *reach* (an LLM can call it), never a
different view: #7 carries §13's revocation bit, #1 never got one.
</details>


`build_chain_mcp(client)` wraps those same callables in a real FastMCP server — the
docstrings become the tool descriptions an LLM reads.

One new construct guards the door. FastMCP's methods are **async**: a function declared
`async def` does *not* run when you call it — the call hands back a **coroutine**, a
promise-of-a-result — and `await` is how you pause until the result is actually ready.
Server libraries are written this way because a real server juggles many callers at
once. Jupyter runs every cell inside an **event loop** (the scheduler that resumes
paused `await`s), so `await` works at the top level here — a notebook nicety, not
general Python. The cell opens with a tiny toy so you see both halves before the real
call:

In [ ]:
async def toy():                     # an async function: CALLING it does not run the body...
    return 7

promise = toy()                      # ...it hands back a coroutine — a promise-of-a-result
print("toy()       →", promise)
print("await toy() →", await promise, " ← await runs it and waits for the value\n")

if not CHAIN:
    print(SKIP)
else:
    try:
        from chainmcp.mcp_server import build_chain_mcp

        server = build_chain_mcp(bell)
        listed = await server.list_tools()          # the same construct, now on a real server method
        for tool in listed:
            print(tool.name.ljust(22), "—", tool.description)
        assert len(listed) == 5
        print("\n✓ the docstrings you can read in mcp_server.py, now tool descriptions for an LLM (→10)")
    except ImportError:
        print("skipped: needs the `mcp` package — `uv sync` installs it")

## 16. Debugging sidebar: BadSignature with everything looking right

The bug you will actually meet, sooner or later: every offer field is correct, the right
key signed, the signature is a healthy 65 bytes — and `fulfill` still says
`BadSignature` (or your `recover` returns an address you've never seen). Remember §5:
**verification never fails loudly, it just points at a stranger.** There is no error
message anywhere, because nothing "errored".

The cause, almost always: the two sides hashed *different bytes* — and the quietest way
for that to happen is **domain drift**. Watch one character do it:

In [ ]:
from eth_account.messages import encode_typed_data

bell_acct = Account.from_key(ANVIL_KEYS["bell"])
msg_712 = offer_to_message(fx.CANONICAL_OFFER)

right = eip712_domain(31337, PRETEND_SETTLEMENT)
sig_712 = bell_acct.sign_typed_data(domain_data=right, message_types=OFFER_TYPES,
                                    message_data=msg_712).signature

wrong = dict(right, version="1")            # ONE character of domain drift
ok  = Account.recover_message(encode_typed_data(domain_data=right, message_types=OFFER_TYPES,
                                                message_data=msg_712), signature=sig_712)
bad = Account.recover_message(encode_typed_data(domain_data=wrong, message_types=OFFER_TYPES,
                                                message_data=msg_712), signature=sig_712)

print("recover, right domain  →", ok, " ✓ Bell" if ok == bell_acct.address else " ✗")
print("recover, version='1'   →", bad, " ← a stranger. no exception. nothing 'failed'.")
assert ok == bell_acct.address and bad != bell_acct.address

So the debugging ritual, in order of likelihood — **diff the domain first**:

1. the four domain fields: `name`, `version`, `chainId`, `verifyingContract` — compare
   both stacks character by character (docs/03 §2.1 is the reference spelling);
2. the type string: field *order* and exact type names (§7);
3. dynamic fields hashed-inside: is `bytes params` entering as its keccak on both sides? (§7)
4. only then suspect the key or the signature bytes themselves.

This repo pre-empts the whole class with §9's cross-stack digest assert, pinned in CI —
which is why it exists.

**✏️ Your turn 16.1 — the domain line-up**

`sig_712` above was made under exactly one of the four candidate domains below. B, C and
D each break a *different* field — before running, write down which field disqualifies
each. Success: exactly one candidate recovers to Bell, and you named all three broken
fields.

In [ ]:
# my prediction — B breaks: ...  C breaks: ...  D breaks: ...      ← TODO fill in first
candidates = {
    "A": eip712_domain(31337, PRETEND_SETTLEMENT),
    "B": dict(eip712_domain(31337, PRETEND_SETTLEMENT), version="1"),
    "C": eip712_domain(1, PRETEND_SETTLEMENT),
    "D": dict(eip712_domain(31337, PRETEND_SETTLEMENT), name="A2Aprovisioning"),
}
for tag, dom in candidates.items():
    who = Account.recover_message(
        encode_typed_data(domain_data=dom, message_types=OFFER_TYPES, message_data=msg_712),
        signature=sig_712,
    )
    print(tag, "→", who, " ✓ Bell" if who == bell_acct.address else "")
# hits = [t for t, d in candidates.items()
#         if Account.recover_message(encode_typed_data(domain_data=d, message_types=OFFER_TYPES,
#                                    message_data=msg_712), signature=sig_712) == bell_acct.address]
# assert hits == ["A"]        # un-comment once you've predicted

<details><summary>✅ Solution 16.1 — peek only after trying</summary>

```python
hits = [t for t, d in candidates.items()
        if Account.recover_message(encode_typed_data(domain_data=d, message_types=OFFER_TYPES,
                                   message_data=msg_712), signature=sig_712) == bell_acct.address]
assert hits == ["A"]
```

B breaks `version` ("1" ≠ "0"), C breaks `chainId` (1 ≠ 31337), D breaks `name` — one
lowercase letter (`A2Aprovisioning`). All three recover to strangers with no error
anywhere; only the byte-exact domain finds Bell. That's why the ritual says *diff the
domain first*.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| scanned the repo for `Account.from_key` | rule 2 machine-checked: signing power only in chainmcp; the controller verifies, never signs | 08 recovers proofs with zero keys |
| built one client per key; `isinstance(ada, EntitlementReader)` | the adapter pattern cashes in: 03's port + 06's chain = the plug | worlds swap fake ↔ real without the controller noticing (11) |
| read `ChainReader`'s borrowed methods | custody by *absence*: composition, not inheritance — no dormant path to a key | the controller's reader in 08 |
| signed & recovered EIP-191 | authenticity · integrity · non-repudiation; recovery names the signer, never errors | activation proofs, verified in pure Python (08) |
| diffed the four domain fields | EIP-712 domain binds a signature to one protocol, version, chain, contract | the §2.1 constants everyone must respect |
| rebuilt the type string; found it in `Settlement.sol` | field order is part of the signature; dynamic fields enter as their keccak | the rule the cross-stack test pins forever |
| `offer_digest == hashOffer`, byte for byte | milestone M1.5: two stacks, one hash | pinned CI test `test_cross_stack.py` |
| approved 3 TOK by hand, then withdrew it | allowance: pull-based payment — permission, not payment | the buy step in every agent purchase (10, 12) |
| bought #7; asserted six effects; `get(7)` == the fixture | atomic settlement (I3), single-use digests (I2), typed views at the border | the predicate's input (08) |
| replayed & tampered → named refusals, no trace | end-to-end integrity: offers are tamper-proof over ANY wire | A2A messages between agents (10) |
| watched `Revoked(7)` arrive on a thread | at-least-once event delivery; chain time, never OS time (ADR-004) | the revocation showpiece (12) |
| decoded `tokenURI` after revoking | on-chain fine print that can't 404 and can't lie (I7) | evidence in the finale (12) |
| listed the five MCP tools | the key as a *service*: tools out, key never | the agent graphs' hands (10) |

## Experiments to try

Predict first, then run:

- **The M1.5 classic, reproduced:** edit `chainmcp/src/chainmcp/signing.py` and change
  the domain `version` to `"1"`. Restart the kernel, rerun the notebook — which cell
  fails *first*? (Not §9: §6's `assert domain["version"] == "0"` reads the domain
  directly and trips long before the seam is exercised. Now comment that assert out
  and rerun — *this* time §9's cross-stack digest assert catches the drift, which is
  the point: the CI-pinned seam check works even for drifts no spelling check
  anticipates.) Change everything back.
- `bell.revoke(7)` a second time — error or success? And does `seen` grow? (Recall
  13.1's solution and rule 8.)
- `sorted(ada._errors.values())` — all 26 refusals the chain can name. Find the three
  that `FakeChain` shares (05), and compute `keccak(text="NotIssuer()")[:4]` by hand to
  match its selector.
- `ada.owner_of(999)` — why is this a `KeyError` and not a `ChainRevert`? (Port parity:
  05's callers were written against a dict. The client's docstring says it out loud.)

## Check yourself

1. Why does `ChainReader` *borrow* ChainClient's methods instead of inheriting from it?
2. What three guarantees does a signature add over a plain hash?
3. Name the four domain fields and the replay each one prevents.
4. Why does `bytes params` enter the struct hash as `keccak256(params)` — and what
   happens if only one side remembers that rule?
5. Why can't `fulfill` just take Ada's TOK — what are the two steps, and which one
   moves money?
6. `fulfill` says `BadSignature` but every offer field checks out. What do you diff
   first, and why will no error message help you?

**Where this goes next:** `08_controller_the_bouncer.ipynb` — the *other* side of both
signatures: a keyless `ChainReader` in the controller's hands, `recover` guarding the
door, and every deny path you met here becoming an HTTP refusal.

**Deeper dive:** [`e2e/notebooks/chain_client_explore.ipynb`](../chain_client_explore.ipynb)
(the compact tour) · [`docs/03-interfaces.md`](../../../docs/03-interfaces.md) §2–§3
(the signing seam, the §2.1 domain, the §3.2 proof) ·
[`docs/03a-interfaces-walkthrough.md`](../../../docs/03a-interfaces-walkthrough.md) ·
[`contracts/EXPLORE-fulfill.md`](../../../contracts/EXPLORE-fulfill.md) (the same
purchase, by hand with `cast`) · `chainmcp/tests/test_cross_stack.py` (this chapter,
pinned in CI).

One last cell, always: stop the watcher threads, then fold the throwaway world away
(rule: whoever launches an anvil stops it — daemon threads polling a dead RPC help
nobody).

In [ ]:
if CHAIN:
    for client in (ada, bell, carol, mallory, reader):
        client.close()                # stop watcher threads BEFORE the chain disappears
    anvil.stop()
    print("world folded away — rerun from the top anytime")
else:
    print("no chain was launched — nothing to fold away")